In [4]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

df['Size']

0                       19M
1                       14M
2                      8.7M
3                       25M
4                      2.8M
                ...        
10836                   53M
10837                  3.6M
10838                  9.5M
10839    Varies with device
10840                   19M
Name: Size, Length: 10841, dtype: object

Reto 1: La Trampa de las Unidades de Medida
El equipo de infraestructura quiere saber si las aplicaciones más pesadas tienen menos descargas. Para averiguarlo, necesitan que la columna Size sea completamente numérica. Sin embargo, si revisan la columna, encontrarán valores como "19M" (Megabytes), "201k" (Kilobytes) y un texto molesto que dice "Varies with device".
Tu misión algorítmica:
Escribe una función pura en Python que reciba un string. Si el string termina en 'M', quítale la 'M' y conviértelo a flotante (dejándolo como Megabytes). Si termina en 'k', quítale la 'k', conviértelo a flotante y divídelo entre 1024 (para pasarlo también a Megabytes). Si dice "Varies with device", devuélvelo como un valor nulo de Numpy (np.nan).
Aplica esta función a toda la columna utilizando el método .apply().
Pregunta a responder: Una vez convertida la columna a valores numéricos (Megabytes), ejecuta el método .mean(). ¿Cuál es el peso promedio en Megabytes de las apps en la Play Store?

## Respuesta:
El peso promedio es de 21.52 MB

In [5]:
def clean_size(size):
    if pd.isna(size) or not isinstance(size, str):
        return np.nan

    if size == 'Varies with device':
        return np.nan

    if size.endswith('M'):
        value = size.replace('M', '')
        return float(value)

    if size.endswith('k'):
        value = size.replace('k', '')
        return float(value) / 1024

    return np.nan

df['Size'] = df['Size'].apply(clean_size)

avg_mb = df['Size'].mean()
print("Promedio peso de descarga: ", round(avg_mb, 2), "MB")

Promedio peso de descarga:  21.52 MB


Reto 2: El Tipo de Dato Cronológico
Ningún análisis de software está completo sin analizar el tiempo. La columna Last Updated tiene fechas escritas como texto: "January 7, 2018". Para un modelo matemático o una serie de tiempo, eso es texto inservible.
Tu misión algorítmica:
Investiga y utiliza la función pd.to_datetime() de Pandas para sobrescribir la columna Last Updated, convirtiéndola del tipo string (Object) al tipo nativo datetime64.
Ahora que es un objeto de tiempo, Pandas te permite extraer componentes específicos. Crea una nueva columna llamada Year_Updated extrayendo únicamente el año (df['Last Updated'].dt.year).
Pregunta a responder: Utilizando la sumarización categórica (value_counts()) sobre tu nueva columna Year_Updated, ¿en qué año se actualizó la mayor cantidad de aplicaciones en nuestro dataset?

## Respuesta:
en 2018 se actualizaron 7349 apps

In [6]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

df['Year Updated'] = df['Last Updated'].dt.year

year_act = df['Year Updated'].value_counts()
print(year_act)

Year Updated
2018.0    7349
2017.0    1867
2016.0     804
2015.0     459
2014.0     209
2013.0     110
2012.0      26
2011.0      15
2010.0       1
Name: count, dtype: int64


Reto 3: La Decisión Arquitectónica
Al resolver el Reto 1, introdujiste intencionalmente valores NaN en las aplicaciones cuyo tamaño decía "Varies with device".
Tu misión algorítmica: Evalúa cuántos registros quedaron vacíos. Como ingenieros, decidan y apliquen la mejor técnica: ¿es matemáticamente más sano borrar esas filas con un .dropna() porque el peso de la app es crítico, o prefieren rellenar ese hueco imputando la mediana global del peso de las apps?
Pregunta a responder: Redacten una breve justificación técnica de 3 líneas explicando qué método eligieron y por qué lo consideran superior para no dañar al modelo de predicción.

## Respuesta:
Me decidi llenar con la mediana porque borrar esas filas seria perder las aplicaciones mas descargadas de la tienda, lo que arrunaria los datos para el modelo. Y tambien porque la mediana representa el peso tipico de una app sin dejarse llevar por los outliers.

In [8]:
nulls = df['Size'].isna().sum()
total = len(df)

print(f"Apps sin tamaño definido: {nulls} de {total}")

median = df['Size'].median()
df['Size'] = df['Size'].fillna(median)

print(f"Mediana aplicada: {median}")
print(f"Nulos restantes: {df['Size'].isna().sum()}")

Apps sin tamaño definido: 1696 de 10841
Mediana aplicada: 13.0
Nulos restantes: 0
